In [1]:
import pandas as pd
import numpy as np
import os
from glob import glob
import datetime

In [2]:
rain_df = pd.read_csv("/mnt/d/SEM_six_SGP/Crop_Price_V2/rain/gujarat_monthly_rainfall_clean_2005_2024_final.csv")
ndvi_df = pd.read_csv("/mnt/d/SEM_six_SGP/Crop_Price_V2/NDVI/gujarat_monthly_ndvi_clean_2005_2024_final.csv")

for df in [rain_df, ndvi_df]:
    df.columns = df.columns.str.strip().str.lower()

In [3]:
def normalize_district(name):
    return (
        str(name)
        .lower()
        .replace(" ", "")
        .replace("(baroda)", "")
        .replace("(", "")
        .replace(")", "")
        .strip()
    )

In [4]:
def normalize_district(name):
    return (
        str(name)
        .lower()
        .replace(" ", "")
        .replace("(baroda)", "")
        .replace("(", "")
        .replace(")", "")
        .strip()
    )

In [5]:
mandi_mapping = {
    "ahmedabad": "ahmadabad",
    "banaskanth": "banaskantha",
    "junagarh": "junagadh",
    "vadodara": "vadodara",
    "vadodarabaroda": "vadodara",
    "thedangs": "thedangs",
    "the dangs": "thedangs",
    "gir somnath": "girsomnath",
    "devbhumi dwarka": "devbhumidwarka",
    "chhota udaipur": "chhotaudaipur",
    "panchmahals": "panchmahals",
    "sabarkantha": "sabarkantha",
    "mehsana": "mahesana",
}

In [6]:
folder_path = r"/mnt/d/SEM_six_SGP/crops/cleaned_data"
all_files = glob(os.path.join(folder_path, "*.csv"))

In [7]:
folder_path = r"/mnt/d/SEM_six_SGP/crops/cleaned_data"
final_folder = "/mnt/d/SEM_six_SGP/Crop_Price_V2/final/"
os.makedirs(final_folder, exist_ok=True)

all_files = glob(os.path.join(folder_path, "*.csv"))

for file in all_files:
    
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip().str.lower()
    
    crop_name = os.path.basename(file).replace(".csv", "").lower()
    
    # Normalize district
    df["district"] = df["district"].apply(normalize_district)
    df["district"] = df["district"].replace(mandi_mapping)
    
    # Convert arrival_date
    df["arrival_date"] = pd.to_datetime(df["arrival_date"], errors="coerce")
    df["year"] = df["arrival_date"].dt.year
    df["month"] = df["arrival_date"].dt.month
    
    # Clean price
    df["modal_price"] = pd.to_numeric(df["modal_price"], errors="coerce")
    df = df[df["modal_price"] > 0]
    
    # Daily → Monthly
    monthly = (
        df.groupby(["district", "commodity", "year", "month"])
        .agg(
            monthly_mean_price=("modal_price", "mean"),
            days_traded=("modal_price", "count")
        )
        .reset_index()
    )
    
    # Merge Rain
    monthly = monthly.merge(
        rain_df,
        on=["district", "year", "month"],
        how="left"
    )
    
    missing_rain = monthly[monthly["monthly_rain_sum"].isna()]
    missing_rain.to_csv(
        f"{final_folder}{crop_name}_missing_rain.csv",
        index=False
    )
    
    # Merge NDVI
    monthly = monthly.merge(
        ndvi_df,
        on=["district", "year", "month"],
        how="left"
    )
    
    missing_ndvi = monthly[monthly["monthly_ndvi_mean"].isna()]
    missing_ndvi.to_csv(
        f"{final_folder}{crop_name}_missing_ndvi.csv",
        index=False
    )
    
    # Save final crop dataset
    monthly.to_csv(
        f"{final_folder}{crop_name}_final.csv",
        index=False
    )
    
    print(f"Processed {crop_name}")

Processed ajwan
Processed alasande_gram
Processed amaranthus
Processed apple
Processed banana
Processed beans
Processed beetroot
Processed big_gram
Processed brinjal
Processed cabbage
Processed capsicum
Processed carrot
Processed castor_seed
Processed cauliflower
Processed chili_red
Processed chilly_capsicum
Processed colacasia
Processed cotton
Processed cotton_seed
Processed drumstick
Processed dry_chillies
Processed field_pea
Processed garlic
Processed grapes
Processed green_chilli
Processed green_peas
Processed groundnut
Processed ground_nut_seed
Processed guar
Processed guava
Processed jute
Processed leafy_vegetable
Processed lime
Processed maize
Processed mango
Processed mataki
Processed methi_seeds
Processed millets
Processed moath_dal
Processed mustard
Processed onion
Processed onion_green
Processed orange
Processed other_pulses
Processed papaya
Processed peas_wet
Processed pineapple
Processed pomegranate
Processed pumpkin
Processed raddish
Processed rajgir
Processed raya
Proces

In [50]:
file = all_files[0]

df_test = pd.read_csv(file)
df_test.columns = df_test.columns.str.strip().str.lower()

print("Columns in file:", os.path.basename(file))
print(df_test.columns)

Columns in file: ajwan.csv
Index(['state', 'district', 'market', 'commodity', 'variety', 'grade',
       'arrival_date', 'min_price', 'max_price', 'modal_price',
       'commodity_code'],
      dtype='object')
